In [4]:
import torch
import torch.optim as optim
from torch.utils.data import DataLoader
from transformers import AutoModel, EsmTokenizer
import os
import pandas as pd
import sys
from tqdm import tqdm
import math
from torch.utils.data import WeightedRandomSampler 
from src.layers import Denoiser
from src.diffusion import Diffusion
from src.data_cfg_balanced import UnifiedCFGDataset, collate_fn_cfg
from src.ema_pytorch import EMA

# --- Config ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ESM_MODEL_NAME = "facebook/esm2_t6_8M_UR50D"
EMBEDDING_DIM = 320
BATCH_SIZE = 128 
DIFFUSION_EPOCHS = 500

TIMESTEPS = 1000 

LEARNING_RATE = 1e-6
WEIGHT_DECAY = 1e-12# 增加正则化

CFG_DROPOUT = 0.2 

SAVE_DIR = "trained_models_new"

# 0: is_amp, 1: is_not_hemo, 2: is_not_hemo_amp
# Uncond is automaticly 3
NUM_CLASSES = 3 

def main():
    print(f"--- Starting Diffusion Training ---")
    os.makedirs(SAVE_DIR, exist_ok=True)
    
    print("Loading data...")
    data_sources = {}

    if os.path.exists("data/antimicrobial.csv"):
        df = pd.read_csv("data/antimicrobial.csv")
        col = 'sequence' if 'sequence' in df.columns else 'SEQUENCE'
        if col in df.columns:
            df = df.rename(columns={col: 'SEQUENCE'})
            data_sources["antimicrobial"] = (df, 0) # 强制 Label 0
    
    if os.path.exists("data/Mixed_data.csv"):
        df_u = pd.read_csv("data/Mixed_data.csv")
        data_sources["unified"] = (df_u, None) # 使用自带 Label
        
    if not data_sources:
        print("Error: No data found.")
        return

    print("Loading ESM...")
    tokenizer = EsmTokenizer.from_pretrained(ESM_MODEL_NAME)
    esm_model = AutoModel.from_pretrained(ESM_MODEL_NAME).to(DEVICE).eval()

    dataset = UnifiedCFGDataset(data_sources, tokenizer, esm_model, DEVICE, augment_prob=0.2)
    all_labels = [int(y) for y in dataset.labels]
    class_counts = {}
    for y in all_labels:
        class_counts[y] = class_counts.get(y, 0) + 1
    
    print(f"Original Class Counts: {class_counts}")
    
    # weight = 1.0 / count
    class_weights = {0:1,1:10,2:5}
    
    sample_weights = [class_weights[y] for y in all_labels]
    sample_weights = torch.DoubleTensor(sample_weights)
    

    sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)
    
    dataloader = DataLoader(
        dataset, 
        batch_size=BATCH_SIZE, 
        shuffle=False, 
        sampler=sampler, 
        collate_fn=collate_fn_cfg, 
        num_workers=0
    )
        
    # --- Model reset---
    print("Initializing Diffusion Architecture...")
    denoiser = Denoiser(esm_model, input_dim=EMBEDDING_DIM, dropout_prob=CFG_DROPOUT).to(DEVICE)
    
    diffusion_model = Diffusion(denoiser, timesteps=TIMESTEPS).to(DEVICE)
    
    optimizer = optim.AdamW(denoiser.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    
    # EMA
    ema = EMA(denoiser, 0.995)
    ema.register()
    
    # --- Training ---
    print("Starting training...")
    
    for epoch in range(DIFFUSION_EPOCHS):
        denoiser.train()
        total_loss = 0
        pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}")
        
        for batch in pbar:
            optimizer.zero_grad()
            
            embeddings = batch['embeddings'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)
            
            loss = diffusion_model.loss(embeddings, labels, mask)
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(denoiser.parameters(), 1.0)
            optimizer.step()
            ema.update()
            
            total_loss += loss.item()
            pbar.set_postfix({"loss": loss.item()})
            
        avg_loss = total_loss / len(dataloader)
        print(f"Epoch {epoch+1} Avg Loss: {avg_loss:.5f}")
        
        # Save
        if (epoch + 1) % 10 == 0:
            ema.apply_shadow()
            torch.save(denoiser.state_dict(), os.path.join(SAVE_DIR, f"epoch_{epoch+1}.pth"))
            ema.restore()

    print("Training Done!")

if __name__ == "__main__":
    main()

--- Starting Diffusion Training ---
Loading data...
Loading ESM...


Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t6_8M_UR50D and are newly initialized: ['esm.pooler.dense.bias', 'esm.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Loading data sources (No Oversampling)...
-> Loaded 22536 sequences from 'antimicrobial' with manual label 0
-> Loaded 6038 sequences from 'unified' using its 'label' column
Final Training Distribution: {0: 23711, 1: 646, 2: 4217}
Total sequences: 28574
Original Class Counts: {0: 23711, 2: 4217, 1: 646}
Initializing Architecture...
Denoiser(
  (time_emb): Sequential(
    (0): SinusoidalPositionEmbeddings()
    (1): Linear(in_features=640, out_features=1280, bias=True)
    (2): SiLU()
    (3): Linear(in_features=1280, out_features=1280, bias=True)
  )
  (label_emb): LabelEmbedder(
    (embedding_table): Embedding(4, 1280)
  )
  (esm_attention_list): ModuleList(
    (0-5): 6 x EsmAttention(
      (self): EsmSelfAttention(
        (query): Linear(in_features=320, out_features=320, bias=True)
        (key): Linear(in_features=320, out_features=320, bias=True)
        (value): Linear(in_features=320, out_features=320, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
        (rota

Epoch 1: 100%|███████████████████| 224/224 [01:43<00:00,  2.17it/s, loss=0.0493]


Epoch 1 Avg Loss: 0.05979


...

Epoch 217 Avg Loss: 0.00923


Epoch 218: 100%|█████████████████| 224/224 [01:38<00:00,  2.27it/s, loss=0.0119]


Epoch 218 Avg Loss: 0.00946


Epoch 219:  31%|█████▌            | 69/224 [00:30<01:08,  2.25it/s, loss=0.0114]


KeyboardInterrupt: 